# Hard-wall max-mix purification: Nx=20, Ny=20,30,40, S=100

This lane uses the support-truncated hard wall. It stages the tested v3 bundle under `/content`, streams nested progress bars, and can import an exact verified v2 checkpoint before resuming.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json
import torch

CONSTRUCTION = 'hard'
REPORT_ONLY = False
MAX_NEW_EXECUTION_BATCHES = None  # Set to 1 for a timing/restart test.
IMPORT_VERIFIED_V2_CHECKPOINTS = True
BUNDLE_DRIVE_DIR = Path('/content/drive/MyDrive/final_production_new_designs/07_maxmix_hard_soft_purification_v3')
OUTPUT_ROOT = Path('/content/drive/MyDrive/classA_final_production_outputs/maxmix_hard_soft_purification_nx20_ny20-40_s100_4ny_raster_v3')
LEGACY_V2_OUTPUT_ROOT = Path('/content/drive/MyDrive/classA_final_production_outputs/maxmix_hard_soft_purification_nx20_ny20-40_s100_4ny_raster_v2')
LOCAL_BUNDLE_DIR = Path('/content/07_maxmix_hard_soft_purification_v3_hard')
SCRATCH_ROOT = Path('/content/purification_v3_scratch_hard')
CONFIG = {
    'sampling_revision': 'maxmix_hard_soft_purification_nx20_ny20-40_s100_4ny_raster_v3',
    'root_seed': 2026090407,
    'Nx': 20,
    'Ny_values': [20, 30, 40],
    'samples_per_case': 100,
    'cycles_multiplier': 4,
    'nshell': 1,
    'alpha_1': 1.0,
    'alpha_2': 30.0,
    'trial_orbitals': 'X',
    'filling_fraction': 0.5,
    'n_a': 0.5,
    'init_mode': 'maxmix',
    'sequence': 'raster_y',
    'perfect_correction': True,
    'postselect': False,
    'dtype': 'complex128',
    'device': 'cuda:0',
    'backend': 'local',
    'state_representation': 'covariance',
    'result_shard_size': 5,
    'segment_cycles': 10,
    'resume_initial_purity_tolerance': 0.50000001,
    'execution_batch_size_by_Ny': {'20': 100, '30': 100, '40': 40},
    'observer_sample_chunk_by_Ny': {'20': 20, '30': 10, '40': 5},
    'constructions': {
        'hard': {'DW': True, 'dw_truncation': True, 'meas_slab_only': True},
        'soft': {'DW': True, 'dw_truncation': False, 'meas_slab_only': False},
    },
    'observables': {
        'cycles': '0..4Ny inclusive',
        'occupation_spectrum': True,
        'cell_entropy_contour': True,
        'cell_charge_variance_contour': True,
        'final_centered_covariance': True,
        'measurement_log_probability': True,
        'cumulative_log_probability': True,
        'log_probability_dtype': 'float64',
        'hard_log_probability_origin': 'after_born_conditioned_exterior_preparation',
        'soft_log_probability_origin': 'global_maxmix_cycle_zero',
        'entropy_log_base': 'natural',
    },
}
if not torch.cuda.is_available():
    raise RuntimeError('This production notebook requires CUDA.')
gpu_name = torch.cuda.get_device_name(0)
gpu_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
if 'A100' not in gpu_name or gpu_gib < 35:
    raise RuntimeError(f'Expected an A100 40-GB-class GPU, got {gpu_name} ({gpu_gib:.2f} GiB)')
if torch.complex128 != getattr(torch, CONFIG['dtype']):
    raise RuntimeError('Locked dtype is not complex128.')
print(json.dumps({
    'construction': CONSTRUCTION, 'config': CONFIG, 'bundle': str(BUNDLE_DRIVE_DIR),
    'output': str(OUTPUT_ROOT), 'scratch': str(SCRATCH_ROOT),
    'gpu': gpu_name, 'gpu_GiB': gpu_gib, 'report_only': REPORT_ONLY,
    'max_new_execution_batches': MAX_NEW_EXECUTION_BATCHES,
    'import_verified_v2_checkpoints': IMPORT_VERIFIED_V2_CHECKPOINTS,
    'legacy_v2_output': str(LEGACY_V2_OUTPUT_ROOT),
}, indent=2, sort_keys=True))

In [ ]:
import codecs
import os
import shutil
import subprocess
import sys

if not BUNDLE_DRIVE_DIR.is_dir():
    raise FileNotFoundError(f'Missing Drive bundle: {BUNDLE_DRIVE_DIR}')
if LOCAL_BUNDLE_DIR.exists():
    shutil.rmtree(LOCAL_BUNDLE_DIR)
shutil.copytree(BUNDLE_DRIVE_DIR, LOCAL_BUNDLE_DIR)
config_path = Path('/content/maxmix_hard_soft_purification_config.json')
config_path.write_text(json.dumps(CONFIG, indent=2, sort_keys=True) + '\n', encoding='utf-8')
command = [
    sys.executable, '-u', str(LOCAL_BUNDLE_DIR / 'run_campaign.py'),
    '--config', str(config_path), '--construction', CONSTRUCTION,
    '--output-root', str(OUTPUT_ROOT), '--scratch-root', str(SCRATCH_ROOT),
]
if REPORT_ONLY:
    command.append('--report-only')
if MAX_NEW_EXECUTION_BATCHES is not None:
    command.extend(['--max-new-execution-batches', str(int(MAX_NEW_EXECUTION_BATCHES))])
if IMPORT_VERIFIED_V2_CHECKPOINTS:
    command.extend(['--import-v2-checkpoints-from', str(LEGACY_V2_OUTPUT_ROOT)])
print('[launch] ' + ' '.join(command), flush=True)
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
decoder = codecs.getincrementaldecoder('utf-8')(errors='replace')
try:
    while True:
        chunk = os.read(process.stdout.fileno(), 8192)
        if not chunk:
            break
        sys.stdout.write(decoder.decode(chunk))
        sys.stdout.flush()
    sys.stdout.write(decoder.decode(b'', final=True))
    sys.stdout.flush()
    returncode = process.wait()
finally:
    if process.poll() is None:
        process.terminate()
        process.wait(timeout=30)
if returncode:
    raise subprocess.CalledProcessError(returncode, command)
print('[notebook] hard-wall runner exited successfully', flush=True)

In [ ]:
from google.colab import runtime
runtime.unassign()
print('done')